# U1 — disponibilidad reportada: histórico ampliado

**Pregunta:** ¿con qué precisión podemos predecir los conteos reportados por tipo de bicicletas convencionales y eléctricas por estación a 5, 10 y 15 minutos?

Este notebook ejecuta **el mismo orquestador del punto de entrada**, desde un kernel limpio: lectura → calidad/Silver → preparación/Gold → entrenamiento/comparación → selección → prueba → guardado/recarga. No descarga ni necesita credenciales. Los 290 archivos y resultados de un día se conservan como prueba preliminar en `notebooks/archivo/` y `data/archivo_preliminar/`.

Inicio recomendado desde la raíz del repositorio:

```bash
.venv/bin/python pyspark/jupyter/UNIDAD1/Dim-Disponibilidad/ejecutar_u1.py --notebook --nueva
```

El lanzador comprueba Docker y monta el contexto público. `Run All` también funciona usando ese contexto ya preparado, sin editar rutas. Las capturas permanecen intactas bajo el montaje `./jupyter:/opt`.


## Configuración de próximas ejecuciones

`--spark-master` del punto de entrada prevalece sobre el JSON. El valor por defecto es `local[4]`; para menos recursos, usar `--spark-master 'local[2]'`. El flujo y esta inspección usan la misma configuración y fábrica de sesión; la memoria no se incrementa.

La ejecución ampliada `20261117T171917344468Z` utilizó **local[2]**. Sus resultados y tiempos están en el [notebook archivado](archivo/01_historico_ampliado_20261117T171917344468Z.ipynb), sin cambios. Este notebook principal se actualizó y sus salidas se limpiaron; no se volvió a ejecutar el flujo completo para verificar este cambio de configuración. La comprobación es pequeña y está documentada en [el informe de configuración](../reports/configuracion/spark_master.md).

En Run All interactivo se utiliza el contexto guardado por el lanzador. Ejecutarlo antes con la opción deseada; una sesión activa con otro master requiere un kernel nuevo. Cambiar código/configuración invalida la reutilización anterior: ejecutar el flujo completo puede iniciar una nueva preparación y entrenamiento.


## Arquitectura y alcance

Se elige **Lambda**: una ruta batch histórica reproducible sirve para auditar y entrenar, mientras una futura ruta de eventos servirá para inferencia en vivo. La disponibilidad puede cambiar sin un viaje —devoluciones, redistribución, mantenimiento—; no equivale a demanda de viajes. Kappa exigiría conservar/reproducir todo como un flujo duradero; el histórico actual es un archivo de snapshots en Storage, no un log Kafka reproducible. Lambda aprovecha esa fuente sin suponer infraestructura de streaming existente.

```text
U1 implementada: GBFS → capturador externo → Storage → descarga incremental → raw
                → lectura PySpark → Silver → Gold → comparación → artefactos
U2 futura:      GBFS → productor → Kafka → Structured Streaming → inferencia
                → PostgreSQL → Grafana (NO IMPLEMENTADA para esta dimensión)
```

No se afirma que el capturador externo esté implementado en esta carpeta. Las dos rutas futuras requerirán conservar el mismo contrato de variables para evitar diferencias batch/streaming.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, sys
from IPython.display import Markdown, SVG, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/flujo_u1.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0,str(ROOT))
from ejecutar_u1 import crear_ejecucion
from src.flujo_u1 import ejecutar
from src.ejecucion import comprobar_artefactos, configuracion, crear_spark
if os.environ.get('U1_RUN_DIR'):
    RUN = Path(os.environ['U1_RUN_DIR'])
else:
    context_file=ROOT/'data/contexto_notebook.json'
    if not context_file.exists():
        raise RuntimeError('Primero ejecutar el punto de entrada con --notebook para verificar el Docker existente')
    RUN=crear_ejecucion(json.loads(context_file.read_text()))
config=configuracion(RUN/'configuracion.json')
print('Ejecución:',RUN.name,'; configuración UTC:',config)
print('Raw:',ROOT/'data/raw','; sin conexión a Supabase')


In [ ]:
# Única implementación del flujo: se reutiliza desde CLI y notebook.
manifest=ejecutar(RUN,nueva=os.environ.get('U1_FORZAR_NUEVA','0')=='1')
assert manifest['estado']=='completada'
SOURCE=RUN if manifest['modo']=='nueva' else RUN.parent/manifest['ejecucion_fuente']
meta=json.loads((SOURCE/'manifiesto.json').read_text())
comprobar_artefactos(SOURCE,meta['artefactos'])
print('Flujo completo:',manifest['estado'],manifest['modo'],'; duración (s):',manifest['duracion_s'])
print('Linaje:',meta['linaje'])
print('Versiones:',meta['versiones'])


In [ ]:
# Evidencia acotada de esquema y ejemplos; el flujo completo ya terminó.
from pyspark.sql import functions as F
from src.lectura import esquema, leer_fuente
sp=crear_spark(config, app_name='U1-esquemas-y-ejemplos')
sp.sparkContext.setLogLevel('ERROR')
for source in ('station_status','station_information','vehicle_types'):
    print('Esquema explícito de entrada:',source,esquema(source).simpleString())
_,catalog=leer_fuente(sp,ROOT/'data/raw','vehicle_types')
catalog.select('vehicle_type_id','form_factor','propulsion_type').distinct().show(truncate=False)
accepted=sp.read.parquet(str(SOURCE/'silver/aceptados'))
print('Esquema real Silver y ejemplos trazables:')
accepted.printSchema()
accepted.select('station_id','captura_utc','last_updated','last_reported','vehicle_types_available',
    'num_bikes_available','num_ebikes_available','q_discrepancia_total','archivo_origen').show(3,truncate=False)
sp.read.parquet(str(SOURCE/'silver/excluidos')).select('station_id','captura_utc','tipo_registro',
    'num_bikes_available','vehicle_types_available','motivos_exclusion').show(3,truncate=False)
gold_view=sp.read.parquet(str(SOURCE/'gold/aceptados'))
print('Esquema real Gold y objetivos observados:')
gold_view.printSchema()
gold_view.select('station_id','captura_utc','f_captura_utc','horizonte_min','horizonte_efectivo_s',
    'y_convencionales','y_electricas','q_discrepancia_total','f_q_discrepancia_total').show(3,truncate=False)
sp.stop()


In [ ]:
def tabla(rows, columns):
    if len(rows)>200:
        raise ValueError('Solo tablas agregadas pequeñas')
    def value(x):
        return f'{x:.8f}' if isinstance(x,float) else str(x)
    text='| '+' | '.join(columns)+' |\n| '+' | '.join(['---']*len(columns))+' |\n'
    text+='\n'.join('| '+' | '.join(value(r.get(c,'')) for c in columns)+' |' for r in rows)
    display(Markdown(text))
analysis=meta['analisis']
print('Fuentes separadas:',meta['fuentes'])
tabla(analysis['cobertura_fecha'],['fecha_captura_utc','archivos','observaciones','estaciones','aceptados','discrepancia_total','primera_utc','ultima_utc'])
print('Frecuencia real por archivo y reloj interno del feed:',analysis['cadencia'])
print('Huecos por estación:',analysis['huecos'])
print('Cobertura de estaciones:',analysis['cobertura_estaciones'])
tabla(analysis['estaciones_menor_cobertura'],['station_id','capturas','elegibles','primera_utc','ultima_utc'])
display(SVG(filename=str(SOURCE/'analisis/cobertura.svg')))


## Calidad y objetivos observacionales

El catálogo real identifica human y electric_assist. Se usan sus conteos explícitos en `vehicle_types_available`, sin restar eléctricas del total. La extensión `num_ebikes_available` y el total se conservan; su discrepancia es un indicador, no una corrección automática ni prueba de cuál conteo es físicamente correcto.

Se conservan observaciones iguales de instantes distintos. Los 450 s son un umbral de frescura/continuidad provisional, justificado en la investigación original y conservado en esta ampliación. Inactividad no equivale a registro inválido. No se rellenan ausencias con cero. Los rechazos permanecen con motivos por objetivo; motivos superpuestos no se suman como filas distintas.


In [ ]:
print('Reconciliación:',meta['conteos']['silver'],meta['conteos']['gold'])
tabla(analysis['exclusiones_silver'],['tipo_registro','aceptado_silver','count'])
print('Distribuciones de los conteos elegibles:',analysis['distribuciones'])
print('Observaciones con cero:',analysis['ceros'])
print('Episodios de cero, duración observada entre capturas:',{t:v['resumen'] for t,v in analysis['episodios_cero'].items()})
tabla(analysis['por_dia_local'],['fecha_nueva_york','dia_semana_iso','observaciones','convencionales_media','electricas_media'])
display(SVG(filename=str(SOURCE/'analisis/por_hora.svg')))


## RDD, evaluación diferida y particiones

El ejercicio RDD cuenta categorías de elegibilidad/discrepancia usando `map` y `reduceByKey`; reconcilia contra `groupBy` de DataFrame. Su salida tiene como máximo seis categorías, sin llevar el histórico al driver.

La demostración mide jobs de un grupo nuevo: construir `select/where/withColumn` y consultar el plan no inicia jobs; `count` sí. La proyección reduce columnas leídas y Catalyst empuja filtros cuando el origen lo admite. `PartitionFilters` demuestra descarte de particiones; no es lo mismo que filtrar filas después de leerlas.


In [ ]:
print(json.dumps(meta['rdd_y_lazy'],indent=2))
print((SOURCE/'analisis/plan_lazy.txt').read_text())
print('Plan físico de Gold al filtrar una fecha UTC:')
print((SOURCE/'analisis/plan_gold_fecha.txt').read_text())
assert meta['rdd_y_lazy']['jobs_antes']==meta['rdd_y_lazy']['jobs_tras_plan']
assert meta['rdd_y_lazy']['jobs_tras_accion']
assert meta['rdd_y_lazy']['rdd_reconcilia_con_dataframe']


## Gold y separación temporal

Se conserva el 9 parcial como contexto retrospectivo, sin usarlo como día de entrenamiento. Los cortes cronológicos y comunes a todas las estaciones se fijaron antes de entrenar: 10–13 train, 14 validación, 15–16 prueba, en días UTC completos. Los objetivos que alcanzan el siguiente periodo se purgan. Las ausencias reales de estaciones no se completan artificialmente.

Las 30 variables usan valores actuales, rezagos por tiempo a 5/10/15 minutos, cambios y estadísticas de 15 minutos estrictamente pasados; más calendario America/New_York. Se conserva UTC. Objetivos por timestamp con tolerancia ±30 s y horizonte efectivo; no se emparejan por posición ni se atraviesan huecos grandes. Calidad inicial y futura se aplican por separado; los indicadores futuros solo evalúan, nunca predicen. Imputación/transformaciones aprendidas se ajustan únicamente en train.


In [ ]:
print('Cortes y parámetros:',meta['configuracion'])
print('Predictores explícitos:',meta['predictores'])
tabla(sorted(meta['conteos']['por_tipo_horizonte_particion'],key=lambda r:(r['tipo'],r['horizonte_min'],r['particion_temporal'])),['tipo','horizonte_min','particion_temporal','count'])
print('Gold está particionado por fecha UTC inicial y horizonte, no por estación.')


## Entrenamiento y comparación

Se comparan persistencia, LR sin regularización y ridge 0.01/0.1, RF de 8 árboles/profundidad 4 y RF de 16/profundidad 6. La tercera regularización prueba el extremo sin penalización, sin búsqueda exhaustiva. Los parámetros están fijados antes de consultar prueba.

Se selecciona por MAE de validación, con RMSE y R² adicionales y desempate determinista publicado. Las predicciones negativas se acotan a cero, sin redondear a enteros. Persistencia no requiere entrenamiento; si gana se conserva. Se guardan todos los candidatos MLlib y, aparte, ganador y mejor referencia MLlib. No se elige un horizonte usando prueba.


In [ ]:
result=json.loads((SOURCE/'modelos/resultado.json').read_text())
frozen=json.loads((SOURCE/'modelos/seleccion_congelada.json').read_text())
tabla(frozen['comparativa_validacion'],['tipo','horizonte_min','candidato','n','MAE','RMSE','R2'])
print('Ajustes MLlib:',meta['ajustes_mllib'])
print('Selecciones congeladas:',meta['selecciones'])
print('Congelación:',frozen['congelada_utc'],'; inicio prueba:',result['lectura_prueba_utc'])
assert frozen['congelada_utc']<result['lectura_prueba_utc']


In [ ]:
tabla(result['metricas_prueba'],['tipo','horizonte_min','candidato','seleccionado','n','MAE','RMSE','R2'])
winners={(r['tipo'],r['horizonte_min']):r['candidato'] for r in meta['metricas']}
subgroups=[{**r,'categoria':r[r['dimension']]} for r in result['subgrupos_prueba'] if r['candidato']==winners[r['tipo'],r['horizonte_min']]]
tabla(subgroups,['tipo','horizonte_min','candidato','dimension','categoria','n','MAE','RMSE','R2'])
tabla(result['resumen_estaciones'],['tipo','horizonte_min','candidato','estaciones','MAE_macro','p50_p95_MAE_estacion'])


## Importancia de variables de Random Forest

Se conserva `featureImportances` de ambos RF por tipo/horizonte, calculada por los modelos ajustados con train. La tabla siguiente muestra sus cinco componentes principales, incluyendo indicadores de ausencia cuando corresponda. No son efectos causales: variables actuales y rezagos están correlacionados y la imputación puede afectar la atribución. La importancia de una variable no acredita exactitud física de la fuente.


In [ ]:
importance=[]
for path in sorted((SOURCE/'modelos/interpretacion').glob('*/*.json')):
    item=json.loads(path.read_text())
    for row in sorted(item['importancias'],key=lambda r:-r['importancia'])[:5]:
        importance.append({'tipo':item['tipo'],'horizonte_min':item['horizonte_min'],'candidato':item['candidato'],**row})
assert len(list((SOURCE/'modelos/interpretacion').glob('*/*.json')))==12
assert len(importance)==60
tabla(importance,['tipo','horizonte_min','candidato','variable_vector','importancia'])


In [ ]:
print('Recarga de ganadores:',meta['recarga_verificada'])
print('Recarga de referencias MLlib:',meta['recarga_referencias_mllib'])
assert len(meta['recarga_verificada'])==len(meta['recarga_referencias_mllib'])==6
assert all(r['max_diferencia']<=1e-10 and r['nulos']==0 for r in meta['recarga_verificada']+meta['recarga_referencias_mllib'])
comprobar_artefactos(SOURCE,meta['artefactos'])
for key in winners:
    tipo,h=key
    display(SVG(filename=str(SOURCE/'modelos'/f'{tipo}_{h}min_observado.svg')))
print('Artefactos y modelos conservados:',SOURCE)
print('Manifest de esta ejecución:',RUN/'manifiesto.json')


## Conclusiones y límites de la evidencia

Las métricas anteriores permiten comparar estos candidatos sobre los días reservados; no se presupone que persistencia gane ni se cambia la selección después de prueba. El error condicionado a cambios evita confundir estabilidad del inventario con capacidad de anticipar movimientos. Las comparaciones con/sin discrepancias se mantienen y no identifican cuál representación refleja mejor la realidad física.

Son siete días UTC completos y un tramo de contexto: no demuestran estacionalidad ni un patrón semanal repetible. Hay muy pocos días independientes, train no incluye fin de semana y la validación es sábado UTC (viernes por la noche y sábado locales); ese cambio de régimen limita las conclusiones. Las decisiones operativas requerirán más fechas y un umbral de error acordado. La prueba preliminar de un día fue inspeccionada y queda separada; no se usa como evaluación final.

**Hallazgo técnico:** contar flags de exclusión solapados como filas descartadas duplica casos; se reconcilian aceptados/excluidos y se conservan razones por objetivo. Un conteo igual entre capturas sigue siendo una observación temporal válida.

**Reflexión técnica:** evaluación diferida permite optimizar el plan antes del trabajo; validar tipos y calidad antes de Gold evita propagar errores a consumidores. Ninguna métrica sola describe el coste operativo: MAE mide error medio, RMSE enfatiza errores grandes, R² depende también de la variación entre estaciones. El estudiante debe poder explicar estas decisiones con sus propias palabras.

Faltan las evidencias personales: capturas con reloj/perfil, publicación e historial real del aporte, pitch y sustentación individual. U2 no está implementada. No se realizan commits ni push.
